# 📘 SQL Foundation for Data Engineering
## Databricks Notebook — Complete SQL Mastery

**What you'll learn:**
- Build a complete e-commerce data warehouse (`techmart_dw`)
- Master SQL from SELECT to Window Functions
- Data Engineering patterns: SCD, MERGE, CDC
- Performance optimization techniques
- Interview-ready problem solving

**Prerequisites:** Databricks Community Edition account

---

---
# 🏗️ Section 1: Database & Dataset Creation

We start by building the **TechMart Data Warehouse** — a realistic e-commerce dataset
with intentional data quality issues that mirror real-world scenarios.

Every future notebook in this series depends on these tables.

In [ ]:
%sql
-- Create our data warehouse database
CREATE DATABASE IF NOT EXISTS techmart_dw;
USE techmart_dw;

## 👥 Customers Table (5200 rows)
Includes intentional quality issues: NULL emails, inconsistent casing, 200 duplicates.

In [ ]:
%sql
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
  customer_id INT, first_name STRING, last_name STRING, email STRING,
  phone STRING, address STRING, city STRING, state STRING, zip_code STRING,
  country STRING, registration_date DATE, customer_segment STRING,
  lifetime_value DECIMAL(12,2), is_active BOOLEAN, metadata STRING,
  created_at TIMESTAMP, updated_at TIMESTAMP
);

INSERT INTO customers
SELECT
  id AS customer_id,
  CASE WHEN id % 20 = 0 THEN UPPER(CONCAT('First', id))
       WHEN id % 15 = 0 THEN LOWER(CONCAT('first', id))
       ELSE CONCAT('First', id) END AS first_name,
  CONCAT('Last', id) AS last_name,
  CASE WHEN id % 12 = 0 THEN NULL
       ELSE CONCAT('user', id, '@', CASE abs(hash(id*3))%4 WHEN 0 THEN 'gmail.com' WHEN 1 THEN 'yahoo.com' WHEN 2 THEN 'outlook.com' ELSE 'company.com' END)
  END AS email,
  CASE WHEN id % 10 = 0 THEN NULL ELSE CONCAT('+1-555-', LPAD(CAST(abs(hash(id*11))%10000 AS STRING),4,'0')) END AS phone,
  CONCAT(CAST(abs(hash(id*7))%9999+1 AS STRING), ' Main St') AS address,
  CASE abs(hash(id*13))%10 WHEN 0 THEN 'New York' WHEN 1 THEN 'Los Angeles' WHEN 2 THEN 'Chicago' WHEN 3 THEN 'Houston' WHEN 4 THEN 'Phoenix' WHEN 5 THEN 'Philadelphia' WHEN 6 THEN 'San Antonio' WHEN 7 THEN 'San Diego' WHEN 8 THEN 'Dallas' ELSE 'Austin' END AS city,
  CASE abs(hash(id*17))%10 WHEN 0 THEN 'NY' WHEN 1 THEN 'CA' WHEN 2 THEN 'IL' WHEN 3 THEN 'TX' WHEN 4 THEN 'AZ' WHEN 5 THEN 'PA' WHEN 6 THEN 'FL' WHEN 7 THEN 'OH' WHEN 8 THEN 'GA' ELSE 'WA' END AS state,
  LPAD(CAST(abs(hash(id*19))%90000+10000 AS STRING),5,'0') AS zip_code,
  'US' AS country,
  date_add('2018-01-01', abs(hash(id*23))%2000) AS registration_date,
  CASE abs(hash(id*29))%4 WHEN 0 THEN 'Premium' WHEN 1 THEN 'Standard' WHEN 2 THEN 'Basic' ELSE 'New' END AS customer_segment,
  CAST(abs(hash(id*31))%50000+100 AS DECIMAL(12,2)) AS lifetime_value,
  CASE WHEN id%8=0 THEN false ELSE true END AS is_active,
  CONCAT('{"source":"', CASE abs(hash(id*37))%4 WHEN 0 THEN 'web' WHEN 1 THEN 'mobile' WHEN 2 THEN 'referral' ELSE 'organic' END, '","loyalty_tier":', CAST(abs(hash(id*41))%5+1 AS STRING), '}') AS metadata,
  CAST(date_add('2018-01-01', abs(hash(id*23))%2000) AS TIMESTAMP) AS created_at,
  current_timestamp() AS updated_at
FROM (SELECT explode(sequence(1, 5000)) AS id)
UNION ALL
SELECT id+5000, CONCAT('First',id), CONCAT('Last',id), CONCAT('user',id,'@gmail.com'),
  CONCAT('+1-555-',LPAD(CAST(abs(hash(id*11))%10000 AS STRING),4,'0')),
  CONCAT(CAST(abs(hash(id*7))%9999+1 AS STRING),' Main St'), 'New York','NY','10001','US',
  date_add('2018-01-01',abs(hash(id*23))%2000), 'Standard',
  CAST(abs(hash(id*31))%50000+100 AS DECIMAL(12,2)), true,
  '{"source":"web","loyalty_tier":1}', current_timestamp(), current_timestamp()
FROM (SELECT explode(sequence(1, 200)) AS id);

## 📦 Products Table (500 rows)
Product catalog with categories, pricing, and JSON specifications.

In [ ]:
%sql
DROP TABLE IF EXISTS products;
CREATE TABLE products (
  product_id INT, product_name STRING, category STRING, subcategory STRING,
  brand STRING, price DECIMAL(10,2), cost DECIMAL(10,2), weight_kg DECIMAL(6,2),
  description STRING, is_active BOOLEAN, launch_date DATE, specifications STRING,
  created_at TIMESTAMP, updated_at TIMESTAMP
);

INSERT INTO products
SELECT id AS product_id,
  CONCAT(CASE abs(hash(id*3))%5 WHEN 0 THEN 'Pro ' WHEN 1 THEN 'Ultra ' WHEN 2 THEN 'Basic ' WHEN 3 THEN 'Elite ' ELSE 'Max ' END, CASE abs(hash(id*7))%8 WHEN 0 THEN 'Laptop' WHEN 1 THEN 'Phone' WHEN 2 THEN 'Tablet' WHEN 3 THEN 'Monitor' WHEN 4 THEN 'Keyboard' WHEN 5 THEN 'Mouse' WHEN 6 THEN 'Headset' ELSE 'Camera' END, ' ', CAST(id AS STRING)) AS product_name,
  CASE abs(hash(id*11))%6 WHEN 0 THEN 'Electronics' WHEN 1 THEN 'Computers' WHEN 2 THEN 'Accessories' WHEN 3 THEN 'Audio' WHEN 4 THEN 'Photography' ELSE 'Gaming' END AS category,
  CASE abs(hash(id*13))%4 WHEN 0 THEN 'Premium' WHEN 1 THEN 'Standard' WHEN 2 THEN 'Budget' ELSE 'Professional' END AS subcategory,
  CASE abs(hash(id*17))%6 WHEN 0 THEN 'TechCorp' WHEN 1 THEN 'DataBrand' WHEN 2 THEN 'SmartGear' WHEN 3 THEN 'ProTech' WHEN 4 THEN 'NexGen' ELSE 'AlphaWare' END AS brand,
  CAST(abs(hash(id*19))%2000+10 AS DECIMAL(10,2)) AS price,
  CAST((abs(hash(id*19))%2000+10)*0.6 AS DECIMAL(10,2)) AS cost,
  CAST(abs(hash(id*23))%200/10.0+0.1 AS DECIMAL(6,2)) AS weight_kg,
  CONCAT('High quality product #', CAST(id AS STRING)) AS description,
  CASE WHEN id%10=0 THEN false ELSE true END AS is_active,
  date_add('2019-01-01', abs(hash(id*29))%1800) AS launch_date,
  CONCAT('{"color":"', CASE abs(hash(id*31))%5 WHEN 0 THEN 'black' WHEN 1 THEN 'silver' WHEN 2 THEN 'white' WHEN 3 THEN 'blue' ELSE 'red' END, '","warranty_months":', CAST(abs(hash(id*37))%36+6 AS STRING), '}') AS specifications,
  current_timestamp() AS created_at, current_timestamp() AS updated_at
FROM (SELECT explode(sequence(1, 500)) AS id);

## 🛒 Orders Table (20000 rows)
Order data with skewed distribution — some customers have many more orders.

In [ ]:
%sql
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
  order_id INT, customer_id INT, order_date DATE, order_timestamp TIMESTAMP,
  status STRING, total_amount DECIMAL(12,2), discount_amount DECIMAL(10,2),
  shipping_cost DECIMAL(8,2), tax_amount DECIMAL(10,2), payment_method STRING,
  shipping_address STRING, order_source STRING, notes STRING,
  partition_date STRING, created_at TIMESTAMP, updated_at TIMESTAMP
);

INSERT INTO orders
SELECT id AS order_id,
  CASE WHEN id%5=0 THEN abs(hash(id*3))%100+1 ELSE abs(hash(id*7))%5000+1 END AS customer_id,
  date_add('2020-01-01', abs(hash(id*11))%1500) AS order_date,
  CAST(date_add('2020-01-01', abs(hash(id*11))%1500) AS TIMESTAMP) AS order_timestamp,
  CASE abs(hash(id*13))%6 WHEN 0 THEN 'completed' WHEN 1 THEN 'shipped' WHEN 2 THEN 'processing' WHEN 3 THEN 'cancelled' WHEN 4 THEN 'returned' ELSE 'pending' END AS status,
  CAST(abs(hash(id*17))%5000+20 AS DECIMAL(12,2)) AS total_amount,
  CAST(abs(hash(id*19))%500 AS DECIMAL(10,2)) AS discount_amount,
  CAST(abs(hash(id*23))%50+5 AS DECIMAL(8,2)) AS shipping_cost,
  CAST(abs(hash(id*17))%5000*0.08 AS DECIMAL(10,2)) AS tax_amount,
  CASE abs(hash(id*29))%5 WHEN 0 THEN 'credit_card' WHEN 1 THEN 'debit_card' WHEN 2 THEN 'paypal' WHEN 3 THEN 'bank_transfer' ELSE 'crypto' END AS payment_method,
  CONCAT(CAST(abs(hash(id*31))%9999+1 AS STRING), ' Delivery Ave') AS shipping_address,
  CASE abs(hash(id*37))%4 WHEN 0 THEN 'web' WHEN 1 THEN 'mobile_app' WHEN 2 THEN 'phone' ELSE 'in_store' END AS order_source,
  CASE WHEN id%20=0 THEN 'Rush delivery requested' ELSE NULL END AS notes,
  date_format(date_add('2020-01-01', abs(hash(id*11))%1500), 'yyyy-MM') AS partition_date,
  CASE WHEN id%50=0 THEN current_timestamp() ELSE CAST(date_add('2020-01-01', abs(hash(id*11))%1500) AS TIMESTAMP) END AS created_at,
  current_timestamp() AS updated_at
FROM (SELECT explode(sequence(1, 20000)) AS id);

## 📋 Order Items Table (50000 rows)
Line-level detail for each order.

In [ ]:
%sql
DROP TABLE IF EXISTS order_items;
CREATE TABLE order_items (
  order_item_id INT, order_id INT, product_id INT,
  quantity INT, unit_price DECIMAL(10,2), discount_pct DECIMAL(5,2), line_total DECIMAL(12,2)
);

INSERT INTO order_items
SELECT id AS order_item_id,
  abs(hash(id*3))%20000+1 AS order_id,
  abs(hash(id*7))%500+1 AS product_id,
  abs(hash(id*11))%10+1 AS quantity,
  CAST(abs(hash(id*13))%2000+10 AS DECIMAL(10,2)) AS unit_price,
  CAST(abs(hash(id*17))%30 AS DECIMAL(5,2)) AS discount_pct,
  CAST((abs(hash(id*11))%10+1)*(abs(hash(id*13))%2000+10)*(1-abs(hash(id*17))%30/100.0) AS DECIMAL(12,2)) AS line_total
FROM (SELECT explode(sequence(1, 50000)) AS id);

## 💳 Payments Table (22000 rows)
Payment records with JSON gateway responses.

In [ ]:
%sql
DROP TABLE IF EXISTS payments;
CREATE TABLE payments (
  payment_id INT, order_id INT, payment_date DATE, amount DECIMAL(12,2),
  payment_method STRING, payment_status STRING, transaction_ref STRING,
  gateway_response STRING, created_at TIMESTAMP
);

INSERT INTO payments
SELECT id AS payment_id,
  abs(hash(id*3))%20000+1 AS order_id,
  date_add('2020-01-01', abs(hash(id*7))%1500) AS payment_date,
  CAST(abs(hash(id*11))%5000+20 AS DECIMAL(12,2)) AS amount,
  CASE abs(hash(id*13))%5 WHEN 0 THEN 'credit_card' WHEN 1 THEN 'debit_card' WHEN 2 THEN 'paypal' WHEN 3 THEN 'bank_transfer' ELSE 'crypto' END AS payment_method,
  CASE abs(hash(id*17))%4 WHEN 0 THEN 'completed' WHEN 1 THEN 'pending' WHEN 2 THEN 'failed' ELSE 'refunded' END AS payment_status,
  CONCAT('TXN-', LPAD(CAST(id AS STRING),8,'0')) AS transaction_ref,
  CONCAT('{"gateway":"stripe","response_code":"', CASE abs(hash(id*19))%3 WHEN 0 THEN '200' WHEN 1 THEN '201' ELSE '400' END, '","auth_code":"AUTH', CAST(abs(hash(id*23))%99999 AS STRING), '"}') AS gateway_response,
  current_timestamp() AS created_at
FROM (SELECT explode(sequence(1, 22000)) AS id);

## 👔 Employees Table (200 rows)
Hierarchical employee data with manager relationships for self-join practice.

In [ ]:
%sql
DROP TABLE IF EXISTS employees;
CREATE TABLE employees (
  employee_id INT, first_name STRING, last_name STRING, email STRING,
  department STRING, job_title STRING, manager_id INT, hire_date DATE,
  salary DECIMAL(10,2), commission_pct DECIMAL(4,2), is_active BOOLEAN, created_at TIMESTAMP
);

INSERT INTO employees
SELECT id AS employee_id,
  CONCAT('Emp', id) AS first_name, CONCAT('Smith', id) AS last_name,
  CONCAT('emp', id, '@techmart.com') AS email,
  CASE abs(hash(id*7))%6 WHEN 0 THEN 'Engineering' WHEN 1 THEN 'Sales' WHEN 2 THEN 'Marketing' WHEN 3 THEN 'Finance' WHEN 4 THEN 'Operations' ELSE 'HR' END AS department,
  CASE abs(hash(id*11))%5 WHEN 0 THEN 'Senior Engineer' WHEN 1 THEN 'Manager' WHEN 2 THEN 'Analyst' WHEN 3 THEN 'Director' ELSE 'Associate' END AS job_title,
  CASE WHEN id<=5 THEN NULL ELSE abs(hash(id*13))%5+1 END AS manager_id,
  date_add('2015-01-01', abs(hash(id*17))%3000) AS hire_date,
  CAST(abs(hash(id*19))%150000+40000 AS DECIMAL(10,2)) AS salary,
  CASE WHEN abs(hash(id*23))%3=0 THEN CAST(abs(hash(id*29))%20/100.0 AS DECIMAL(4,2)) ELSE NULL END AS commission_pct,
  CASE WHEN id%15=0 THEN false ELSE true END AS is_active,
  current_timestamp() AS created_at
FROM (SELECT explode(sequence(1, 200)) AS id);

## 📊 Inventory Table (1500 rows)
Warehouse inventory snapshots.

In [ ]:
%sql
DROP TABLE IF EXISTS inventory;
CREATE TABLE inventory (
  inventory_id INT, product_id INT, warehouse_id INT, quantity_on_hand INT,
  quantity_reserved INT, reorder_point INT, last_restock_date DATE,
  snapshot_date DATE, created_at TIMESTAMP
);

INSERT INTO inventory
SELECT id AS inventory_id,
  abs(hash(id*3))%500+1 AS product_id, abs(hash(id*7))%5+1 AS warehouse_id,
  abs(hash(id*11))%1000 AS quantity_on_hand, abs(hash(id*13))%100 AS quantity_reserved,
  abs(hash(id*17))%50+10 AS reorder_point,
  date_add('2023-01-01', abs(hash(id*19))%365) AS last_restock_date,
  date_add('2024-01-01', abs(hash(id*23))%180) AS snapshot_date,
  current_timestamp() AS created_at
FROM (SELECT explode(sequence(1, 1500)) AS id);

## 🚚 Shipments Table (18000 rows)
Shipping and delivery tracking data.

In [ ]:
%sql
DROP TABLE IF EXISTS shipments;
CREATE TABLE shipments (
  shipment_id INT, order_id INT, carrier STRING, tracking_number STRING,
  ship_date DATE, estimated_delivery DATE, actual_delivery DATE,
  status STRING, weight_kg DECIMAL(6,2), shipping_cost DECIMAL(8,2), created_at TIMESTAMP
);

INSERT INTO shipments
SELECT id AS shipment_id, abs(hash(id*3))%20000+1 AS order_id,
  CASE abs(hash(id*7))%4 WHEN 0 THEN 'FedEx' WHEN 1 THEN 'UPS' WHEN 2 THEN 'USPS' ELSE 'DHL' END AS carrier,
  CONCAT('TRK', LPAD(CAST(id AS STRING),10,'0')) AS tracking_number,
  date_add('2020-01-01', abs(hash(id*11))%1500) AS ship_date,
  date_add('2020-01-01', abs(hash(id*11))%1500+abs(hash(id*13))%7+2) AS estimated_delivery,
  CASE WHEN id%10=0 THEN NULL ELSE date_add('2020-01-01', abs(hash(id*11))%1500+abs(hash(id*17))%10+1) END AS actual_delivery,
  CASE abs(hash(id*19))%4 WHEN 0 THEN 'delivered' WHEN 1 THEN 'in_transit' WHEN 2 THEN 'out_for_delivery' ELSE 'returned' END AS status,
  CAST(abs(hash(id*23))%500/10.0+0.5 AS DECIMAL(6,2)) AS weight_kg,
  CAST(abs(hash(id*29))%100+5 AS DECIMAL(8,2)) AS shipping_cost,
  current_timestamp() AS created_at
FROM (SELECT explode(sequence(1, 18000)) AS id);

## 🌐 Website Events Table (100000 rows)
Clickstream data for sessionization and funnel analysis.

In [ ]:
%sql
DROP TABLE IF EXISTS website_events;
CREATE TABLE website_events (
  event_id INT, session_id STRING, customer_id INT, event_type STRING,
  page_url STRING, referrer STRING, device_type STRING, browser STRING,
  event_timestamp TIMESTAMP, event_properties STRING, event_date DATE
);

INSERT INTO website_events
SELECT id AS event_id,
  CONCAT('sess_', CAST(abs(hash(id*3))%30000 AS STRING)) AS session_id,
  CASE WHEN id%5=0 THEN NULL ELSE abs(hash(id*7))%5000+1 END AS customer_id,
  CASE abs(hash(id*11))%6 WHEN 0 THEN 'page_view' WHEN 1 THEN 'click' WHEN 2 THEN 'add_to_cart' WHEN 3 THEN 'purchase' WHEN 4 THEN 'search' ELSE 'scroll' END AS event_type,
  CONCAT('/products/', CASE abs(hash(id*13))%5 WHEN 0 THEN 'electronics' WHEN 1 THEN 'computers' WHEN 2 THEN 'accessories' WHEN 3 THEN 'deals' ELSE 'new-arrivals' END) AS page_url,
  CASE abs(hash(id*17))%5 WHEN 0 THEN 'google.com' WHEN 1 THEN 'facebook.com' WHEN 2 THEN 'direct' WHEN 3 THEN 'email' ELSE 'twitter.com' END AS referrer,
  CASE abs(hash(id*19))%3 WHEN 0 THEN 'desktop' WHEN 1 THEN 'mobile' ELSE 'tablet' END AS device_type,
  CASE abs(hash(id*23))%4 WHEN 0 THEN 'Chrome' WHEN 1 THEN 'Firefox' WHEN 2 THEN 'Safari' ELSE 'Edge' END AS browser,
  CAST(date_add('2024-01-01', abs(hash(id*29))%180) AS TIMESTAMP) AS event_timestamp,
  CONCAT('{"duration_sec":', CAST(abs(hash(id*31))%300 AS STRING), '}') AS event_properties,
  date_add('2024-01-01', abs(hash(id*29))%180) AS event_date
FROM (SELECT explode(sequence(1, 100000)) AS id);

## 📝 App Logs Table (50000 rows)
Application logs for error monitoring and debugging.

In [ ]:
%sql
DROP TABLE IF EXISTS app_logs;
CREATE TABLE app_logs (
  log_id INT, log_timestamp TIMESTAMP, log_level STRING, service_name STRING,
  message STRING, error_details STRING, trace_id STRING, user_id INT, log_date DATE
);

INSERT INTO app_logs
SELECT id AS log_id,
  CAST(date_add('2024-01-01', abs(hash(id*3))%180) AS TIMESTAMP) AS log_timestamp,
  CASE abs(hash(id*7))%5 WHEN 0 THEN 'ERROR' WHEN 1 THEN 'WARN' WHEN 2 THEN 'INFO' WHEN 3 THEN 'DEBUG' ELSE 'INFO' END AS log_level,
  CASE abs(hash(id*11))%5 WHEN 0 THEN 'auth-service' WHEN 1 THEN 'payment-service' WHEN 2 THEN 'order-service' WHEN 3 THEN 'inventory-service' ELSE 'notification-service' END AS service_name,
  CASE abs(hash(id*13))%6 WHEN 0 THEN 'Connection timeout' WHEN 1 THEN 'Request processed successfully' WHEN 2 THEN 'Invalid input received' WHEN 3 THEN 'Cache miss' WHEN 4 THEN 'Rate limit exceeded' ELSE 'Health check passed' END AS message,
  CASE WHEN abs(hash(id*7))%5=0 THEN CONCAT('{"stack_trace":"NullPointerException at line ', CAST(abs(hash(id*17))%500 AS STRING), '"}') ELSE NULL END AS error_details,
  CONCAT('trace-', LPAD(CAST(abs(hash(id*19))%10000 AS STRING),6,'0')) AS trace_id,
  abs(hash(id*23))%5000+1 AS user_id,
  date_add('2024-01-01', abs(hash(id*3))%180) AS log_date
FROM (SELECT explode(sequence(1, 50000)) AS id);

## 💰 Transactions Table (30000 rows)
Financial transactions for reconciliation and balance analysis.

In [ ]:
%sql
DROP TABLE IF EXISTS transactions;
CREATE TABLE transactions (
  transaction_id INT, account_id INT, transaction_type STRING, amount DECIMAL(12,2),
  balance_after DECIMAL(14,2), transaction_date DATE, description STRING,
  category STRING, is_reconciled BOOLEAN, reconciled_date DATE, created_at TIMESTAMP
);

INSERT INTO transactions
SELECT id AS transaction_id,
  abs(hash(id*3))%1000+1 AS account_id,
  CASE abs(hash(id*7))%4 WHEN 0 THEN 'credit' WHEN 1 THEN 'debit' WHEN 2 THEN 'transfer' ELSE 'refund' END AS transaction_type,
  CAST(abs(hash(id*11))%10000+1 AS DECIMAL(12,2)) AS amount,
  CAST(abs(hash(id*13))%100000+1000 AS DECIMAL(14,2)) AS balance_after,
  date_add('2022-01-01', abs(hash(id*17))%900) AS transaction_date,
  CONCAT('Transaction #', CAST(id AS STRING)) AS description,
  CASE abs(hash(id*19))%6 WHEN 0 THEN 'salary' WHEN 1 THEN 'utilities' WHEN 2 THEN 'shopping' WHEN 3 THEN 'food' WHEN 4 THEN 'entertainment' ELSE 'other' END AS category,
  CASE WHEN abs(hash(id*23))%4=0 THEN false ELSE true END AS is_reconciled,
  CASE WHEN abs(hash(id*23))%4=0 THEN NULL ELSE date_add('2022-01-01', abs(hash(id*17))%900+5) END AS reconciled_date,
  current_timestamp() AS created_at
FROM (SELECT explode(sequence(1, 30000)) AS id);

## ✅ Verify All Tables

In [ ]:
%sql
SELECT 'customers' AS tbl, COUNT(*) AS rows FROM customers
UNION ALL SELECT 'products', COUNT(*) FROM products
UNION ALL SELECT 'orders', COUNT(*) FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL SELECT 'payments', COUNT(*) FROM payments
UNION ALL SELECT 'employees', COUNT(*) FROM employees
UNION ALL SELECT 'inventory', COUNT(*) FROM inventory
UNION ALL SELECT 'shipments', COUNT(*) FROM shipments
UNION ALL SELECT 'website_events', COUNT(*) FROM website_events
UNION ALL SELECT 'app_logs', COUNT(*) FROM app_logs
UNION ALL SELECT 'transactions', COUNT(*) FROM transactions
ORDER BY tbl;

---
# 📗 Section 2: Beginner SQL

## SELECT, Expressions & Aliases

**WHY:** SELECT is the foundation of every query. Understanding expressions and aliases
lets you transform data on the fly without changing the underlying table.

**Key concepts:**
- `SELECT *` retrieves all columns (avoid in production — specify columns)
- Expressions compute new values from existing columns
- Aliases (`AS`) give readable names to computed columns

In [ ]:
%sql
-- Basic SELECT with specific columns
SELECT customer_id, first_name, last_name, email, customer_segment
FROM customers
LIMIT 10;

In [ ]:
%sql
-- Expressions and aliases: compute profit margin
SELECT 
  product_id, product_name, price, cost,
  price - cost AS profit,
  ROUND((price - cost) / price * 100, 2) AS profit_margin_pct
FROM products
LIMIT 10;

In [ ]:
%sql
-- String concatenation and formatting
SELECT 
  customer_id,
  CONCAT(first_name, ' ', last_name) AS full_name,
  UPPER(email) AS email_upper,
  LENGTH(first_name) AS name_length
FROM customers
LIMIT 10;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Select & Expressions
-- ============================================
-- Write a query that shows:
-- 1. order_id, total_amount, discount_amount
-- 2. A computed column 'net_amount' = total_amount - discount_amount
-- 3. A computed column 'effective_discount_pct' = discount_amount / total_amount * 100
-- Show only the first 15 rows
--
-- Expected: 5 columns, 15 rows
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
SELECT 
  order_id, total_amount, discount_amount,
  total_amount - discount_amount AS net_amount,
  ROUND(discount_amount / total_amount * 100, 2) AS effective_discount_pct
FROM orders
LIMIT 15;

## WHERE — Filtering Rows

**WHY:** WHERE filters rows BEFORE they reach your result set. It's the most
fundamental way to narrow down data.

**Comparison operators:** `=`, `!=` (or `<>`), `<`, `>`, `<=`, `>=`

In [ ]:
%sql
-- Filter by equality and comparison
SELECT order_id, customer_id, total_amount, status
FROM orders
WHERE total_amount > 3000 AND status = 'completed'
LIMIT 10;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: WHERE Clause
-- ============================================
-- Find all products where:
-- 1. price > 500
-- 2. category = 'Electronics'
-- 3. is_active = true
-- Show product_name, price, category
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
SELECT product_name, price, category
FROM products
WHERE price > 500 AND category = 'Electronics' AND is_active = true;

## DISTINCT — Removing Duplicates

**WHY:** Real data has duplicates. DISTINCT helps you find unique values.
Multi-column DISTINCT finds unique combinations.

**Performance note:** DISTINCT requires sorting/hashing all rows — expensive on large tables.

In [ ]:
%sql
SELECT DISTINCT customer_segment FROM customers ORDER BY customer_segment;

In [ ]:
%sql
SELECT DISTINCT category, subcategory FROM products ORDER BY category, subcategory;

## ORDER BY — Sorting Results

**WHY:** SQL results have no guaranteed order without ORDER BY.

**Key points:**
- `ASC` (default), `DESC` = descending
- `NULLS FIRST` / `NULLS LAST` controls NULL placement
- Multi-column: sorts by first, breaks ties with second

In [ ]:
%sql
SELECT customer_id, first_name, email, lifetime_value
FROM customers
ORDER BY lifetime_value DESC NULLS LAST, first_name ASC
LIMIT 15;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: ORDER BY
-- ============================================
-- Find the 20 most recently hired employees
-- Show employee_id, first_name, department, hire_date, salary
-- Sort by hire_date DESC, then salary DESC
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
SELECT employee_id, first_name, department, hire_date, salary
FROM employees
ORDER BY hire_date DESC, salary DESC
LIMIT 20;

## 🎯 Practice: SELECT Fundamentals

Try these exercises on your own before checking the solutions.

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Basic SELECT
-- ============================================
-- 1. Select all columns from the products table, limit to 10 rows
-- 2. Select only product_name, category, and price from products
-- 3. Create a calculated column: price * 1.1 AS price_with_tax
-- 4. Use CONCAT to create a full_name from first_name and last_name in customers
--
-- Write your queries below:


In [ ]:
%sql
-- ✅ SOLUTION 1: All columns, limit 10
SELECT * FROM techmart_dw.products LIMIT 10

In [ ]:
%sql
-- ✅ SOLUTION 2: Specific columns
SELECT product_name, category, price 
FROM techmart_dw.products 
LIMIT 10

In [ ]:
%sql
-- ✅ SOLUTION 3: Calculated column
SELECT product_name, price, 
       ROUND(price * 1.1, 2) AS price_with_tax,
       ROUND(price * 0.1, 2) AS tax_amount
FROM techmart_dw.products 
ORDER BY price DESC
LIMIT 10

In [ ]:
%sql
-- ✅ SOLUTION 4: String concatenation
SELECT customer_id,
       CONCAT(first_name, ' ', last_name) AS full_name,
       email,
       customer_segment
FROM techmart_dw.customers
LIMIT 10

## ⚠️ Common Mistakes: SELECT

| Mistake | Problem | Fix |
|---------|---------|-----|
| `SELECT *` in production | Reads ALL columns (slow, expensive) | Select only needed columns |
| Missing alias for expressions | Column named `price * 1.1` (ugly) | Always use `AS alias_name` |
| Forgetting LIMIT during exploration | Returns millions of rows | Always LIMIT during dev |
| Case sensitivity | `'NYC'` ≠ `'nyc'` in comparisons | Use `LOWER()` or `UPPER()` |

## Advanced WHERE Patterns

Beyond basic comparisons, WHERE supports powerful filtering:

In [ ]:
%sql
-- Multiple conditions with AND/OR (watch operator precedence!)
-- ⚠️ AND binds tighter than OR!

-- This finds: (gold customers) OR (any customer with LTV > 5000)
SELECT customer_id, first_name, customer_segment, lifetime_value
FROM techmart_dw.customers
WHERE customer_segment = 'gold' OR lifetime_value > 5000
LIMIT 10

In [ ]:
%sql
-- Use parentheses to control precedence
-- This finds: gold customers who ALSO have LTV > 5000
SELECT customer_id, first_name, customer_segment, lifetime_value
FROM techmart_dw.customers
WHERE customer_segment = 'gold' AND lifetime_value > 5000
LIMIT 10

In [ ]:
%sql
-- NULL handling in WHERE (the #1 source of bugs!)
-- ⚠️ NULL is NOT equal to anything, not even itself!

-- This will MISS rows where email is NULL:
-- WHERE email != 'test@example.com'  ← NULLs excluded!

-- Correct way to include NULLs:
SELECT customer_id, first_name, email
FROM techmart_dw.customers
WHERE email IS NULL
LIMIT 10

In [ ]:
%sql
-- COALESCE — replace NULL with a default value
SELECT customer_id, 
       first_name,
       COALESCE(email, 'no_email@unknown.com') AS email_safe,
       COALESCE(phone, 'N/A') AS phone_safe
FROM techmart_dw.customers
WHERE email IS NULL OR phone IS NULL
LIMIT 10

---
# 📗 Section 3: Filtering & Logic

## AND / OR / NOT — Logical Operators

**WHY:** Complex business rules require combining conditions.
Operator precedence: NOT > AND > OR. Use parentheses to be explicit.

**Common mistake:** `WHERE status = 'A' OR status = 'B' AND amount > 100`
evaluates as `status = 'A' OR (status = 'B' AND amount > 100)` — probably not what you want!

In [ ]:
%sql
-- AND/OR with explicit parentheses
SELECT order_id, customer_id, total_amount, status, payment_method
FROM orders
WHERE (status = 'completed' OR status = 'shipped')
  AND total_amount > 2000
  AND payment_method != 'crypto'
LIMIT 10;

## IN, BETWEEN — Range & Set Filtering

**WHY:** IN replaces multiple OR conditions. BETWEEN is inclusive on both ends.

In [ ]:
%sql
SELECT order_id, order_date, total_amount
FROM orders
WHERE order_date BETWEEN '2023-01-01' AND '2023-06-30'
  AND total_amount BETWEEN 1000 AND 3000
LIMIT 10;

## LIKE — Pattern Matching

**WHY:** `%` matches any characters, `_` matches exactly one.

**Performance warning:** Leading wildcards (`%text`) cannot use indexes — full table scan!

In [ ]:
%sql
SELECT customer_id, first_name, email
FROM customers
WHERE email LIKE '%gmail.com'
LIMIT 10;

## CASE WHEN — Conditional Logic

**WHY:** CASE WHEN is SQL's if/else. Essential for categorizing data, conditional aggregation, and transformation.

In [ ]:
%sql
SELECT order_id, total_amount,
  CASE
    WHEN total_amount >= 4000 THEN 'Large'
    WHEN total_amount >= 2000 THEN 'Medium'
    WHEN total_amount >= 500 THEN 'Small'
    ELSE 'Micro'
  END AS order_size,
  CASE status
    WHEN 'completed' THEN 'Done'
    WHEN 'cancelled' THEN 'Cancelled'
    WHEN 'returned' THEN 'Returned'
    ELSE 'In Progress'
  END AS status_display
FROM orders LIMIT 15;

## NULL Handling — The Billion Dollar Mistake

**WHY:** NULL means "unknown" — not zero, not empty string.

**Rules:**
- `NULL = NULL` → NULL (not TRUE!)
- Use `IS NULL` / `IS NOT NULL` for checks
- `COALESCE(a, b, c)` returns first non-NULL value
- `NULLIF(a, b)` returns NULL if a = b

In [ ]:
%sql
SELECT customer_id, first_name, email,
  COALESCE(email, 'no_email@unknown.com') AS email_safe,
  COALESCE(phone, 'N/A') AS phone_safe,
  CASE WHEN email IS NULL THEN 'Missing' ELSE 'Present' END AS email_status
FROM customers
WHERE email IS NULL OR phone IS NULL
LIMIT 15;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Filtering & Logic
-- ============================================
-- Find orders where:
-- 1. status NOT IN ('cancelled', 'returned')
-- 2. total_amount BETWEEN 1000 AND 5000
-- 3. order_source starts with 'web' or 'mobile'
-- 4. Add 'priority' column: >4000='High', >2000='Medium', else='Low'
-- Show first 20 rows
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
SELECT order_id, total_amount, status, order_source,
  CASE WHEN total_amount > 4000 THEN 'High'
       WHEN total_amount > 2000 THEN 'Medium'
       ELSE 'Low' END AS priority
FROM orders
WHERE status NOT IN ('cancelled', 'returned')
  AND total_amount BETWEEN 1000 AND 5000
  AND (order_source LIKE 'web%' OR order_source LIKE 'mobile%')
LIMIT 20;

---
# 📗 Section 4: Aggregations

## COUNT, SUM, AVG, MIN, MAX

**Critical distinction:**
- `COUNT(*)` — counts ALL rows (including NULLs)
- `COUNT(column)` — counts non-NULL values only
- `COUNT(DISTINCT column)` — counts unique non-NULL values

In [ ]:
%sql
SELECT 
  COUNT(*) AS total_customers,
  COUNT(email) AS with_email,
  COUNT(DISTINCT email) AS unique_emails,
  COUNT(*) - COUNT(email) AS missing_emails
FROM customers;

In [ ]:
%sql
SELECT 
  COUNT(*) AS total_orders,
  SUM(total_amount) AS revenue,
  ROUND(AVG(total_amount), 2) AS avg_order_value,
  MIN(total_amount) AS smallest_order,
  MAX(total_amount) AS largest_order
FROM orders WHERE status = 'completed';

## GROUP BY & HAVING

**Execution order:** FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY

HAVING filters groups AFTER aggregation (WHERE filters rows BEFORE).

In [ ]:
%sql
SELECT payment_method,
  COUNT(*) AS order_count,
  ROUND(SUM(total_amount), 2) AS total_revenue,
  ROUND(AVG(total_amount), 2) AS avg_order_value
FROM orders WHERE status = 'completed'
GROUP BY payment_method
ORDER BY total_revenue DESC;

In [ ]:
%sql
-- HAVING: high-value customers
SELECT customer_id, COUNT(*) AS order_count, SUM(total_amount) AS total_spent
FROM orders WHERE status != 'cancelled'
GROUP BY customer_id
HAVING COUNT(*) >= 5 AND SUM(total_amount) > 10000
ORDER BY total_spent DESC LIMIT 20;

In [ ]:
%sql
-- Conditional aggregation: pivot-style
SELECT 
  date_format(order_date, 'yyyy-MM') AS month,
  COUNT(*) AS total_orders,
  SUM(CASE WHEN status='completed' THEN 1 ELSE 0 END) AS completed,
  SUM(CASE WHEN status='cancelled' THEN 1 ELSE 0 END) AS cancelled,
  ROUND(SUM(CASE WHEN status='completed' THEN total_amount ELSE 0 END), 2) AS completed_revenue
FROM orders
GROUP BY date_format(order_date, 'yyyy-MM')
ORDER BY month LIMIT 20;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Sales Dashboard Query
-- ============================================
-- For each product category show:
-- 1. Number of distinct products
-- 2. Average price
-- 3. Total items sold (from order_items)
-- 4. Total revenue (sum of line_total)
-- Only categories with > 5000 items sold
-- Hint: JOIN products with order_items
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
SELECT p.category,
  COUNT(DISTINCT p.product_id) AS num_products,
  ROUND(AVG(p.price), 2) AS avg_price,
  COUNT(oi.order_item_id) AS items_sold,
  ROUND(SUM(oi.line_total), 2) AS total_revenue
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY p.category
HAVING COUNT(oi.order_item_id) > 5000
ORDER BY total_revenue DESC;

## 🎯 Practice: Aggregation Challenges

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Aggregation Practice
-- ============================================
-- 1. Find the total revenue per month (use DATE_TRUNC)
-- 2. Find the top 5 customers by total spending
-- 3. Count orders by status, showing percentage of total
-- 4. Find products that have never been ordered (use LEFT JOIN + IS NULL)
--
-- Write your queries below:


In [ ]:
%sql
-- ✅ SOLUTION 1: Monthly revenue
SELECT 
    DATE_TRUNC('month', order_date) AS month,
    COUNT(*) AS total_orders,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS avg_order_value
FROM techmart_dw.orders
WHERE status IN ('completed', 'shipped')
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month DESC
LIMIT 12

In [ ]:
%sql
-- ✅ SOLUTION 2: Top 5 customers by spending
SELECT 
    c.customer_id,
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    c.customer_segment,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(o.total_amount), 2) AS total_spent,
    ROUND(AVG(o.total_amount), 2) AS avg_order_value
FROM techmart_dw.customers c
JOIN techmart_dw.orders o ON c.customer_id = o.customer_id
WHERE o.status IN ('completed', 'shipped')
GROUP BY c.customer_id, c.first_name, c.last_name, c.customer_segment
ORDER BY total_spent DESC
LIMIT 5

In [ ]:
%sql
-- ✅ SOLUTION 3: Orders by status with percentage
SELECT 
    status,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM techmart_dw.orders
GROUP BY status
ORDER BY order_count DESC

In [ ]:
%sql
-- ✅ SOLUTION 4: Products never ordered
SELECT p.product_id, p.product_name, p.category, p.price
FROM techmart_dw.products p
LEFT JOIN techmart_dw.order_items oi ON p.product_id = oi.product_id
WHERE oi.order_item_id IS NULL
ORDER BY p.price DESC

## ⚠️ Common Aggregation Mistakes

| Mistake | Example | Problem | Fix |
|---------|---------|---------|-----|
| COUNT(*) vs COUNT(col) | `COUNT(email)` | Excludes NULLs! | Use `COUNT(*)` for all rows |
| Non-aggregated in SELECT | `SELECT name, SUM(amount)` | Which name? | Add to GROUP BY |
| WHERE vs HAVING | `WHERE SUM(x) > 100` | Can't use agg in WHERE | Use `HAVING SUM(x) > 100` |
| AVG with NULLs | `AVG(discount)` | NULLs excluded from avg | Use `AVG(COALESCE(discount, 0))` |
| Division by zero | `total / count` | Crashes if count=0 | Use `total / NULLIF(count, 0)` |

---
# 📗 Section 5: Joins

## INNER JOIN — Matching Records Only

**Business use:** Get order details with customer names — only orders that have a matching customer.

In [ ]:
%sql
SELECT o.order_id, c.first_name, c.last_name, c.customer_segment, o.total_amount, o.order_date
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
ORDER BY o.total_amount DESC LIMIT 10;

## LEFT JOIN — Finding Missing Data

**Business use:** Find customers who never ordered, products never sold.

In [ ]:
%sql
-- Customers with no orders
SELECT c.customer_id, c.first_name, c.registration_date, COUNT(o.order_id) AS order_count
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.first_name, c.registration_date
HAVING COUNT(o.order_id) = 0
LIMIT 10;

## SELF JOIN — Hierarchical Data

A table joined to itself. Classic use: employee-manager relationships.

In [ ]:
%sql
SELECT e.employee_id, e.first_name AS employee_name, e.department, e.job_title,
  m.first_name AS manager_name, m.job_title AS manager_title
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id
LIMIT 15;

## CROSS JOIN — Date Spine Generation

Produces every combination. Useful for ensuring every date appears in reports.

In [ ]:
%sql
SELECT d.report_date, s.status, COALESCE(oc.cnt, 0) AS order_count
FROM (SELECT date_add('2024-01-01', id-1) AS report_date FROM (SELECT explode(sequence(1,30)) AS id)) d
CROSS JOIN (SELECT DISTINCT status FROM orders) s
LEFT JOIN (
  SELECT order_date, status, COUNT(*) AS cnt FROM orders
  WHERE order_date BETWEEN '2024-01-01' AND '2024-01-30'
  GROUP BY order_date, status
) oc ON d.report_date = oc.order_date AND s.status = oc.status
ORDER BY d.report_date, s.status LIMIT 30;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Multi-Table Join (5 tables)
-- ============================================
-- Join: orders, customers, order_items, products, payments
-- Show: order_id, customer full name, product_name, quantity, line_total, payment_status
-- Filter: completed orders from 2023
-- Limit 20
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
SELECT o.order_id, CONCAT(c.first_name,' ',c.last_name) AS customer_name,
  p.product_name, oi.quantity, oi.line_total, pay.payment_status
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN payments pay ON o.order_id = pay.order_id
WHERE o.status = 'completed' AND o.order_date >= '2023-01-01' AND o.order_date < '2024-01-01'
LIMIT 20;

## 🎯 Practice: JOIN Challenges

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Multi-Table JOIN
-- ============================================
-- Build a COMPLETE order report by joining 5 tables:
-- orders + customers + order_items + products + payments
--
-- Output columns:
-- order_id, order_date, customer_name, product_name, 
-- quantity, unit_price, line_total, payment_method, payment_status
--
-- Filter: only completed orders from the last 6 months
-- Sort: by order_date DESC
-- Limit: 20 rows
--
-- Write your query below:


In [ ]:
%sql
-- ✅ SOLUTION: 5-table JOIN
SELECT 
    o.order_id,
    o.order_date,
    CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
    c.customer_segment,
    p.product_name,
    p.category,
    oi.quantity,
    oi.unit_price,
    oi.line_total,
    pay.payment_method,
    pay.payment_status
FROM techmart_dw.orders o
JOIN techmart_dw.customers c ON o.customer_id = c.customer_id
JOIN techmart_dw.order_items oi ON o.order_id = oi.order_id
JOIN techmart_dw.products p ON oi.product_id = p.product_id
LEFT JOIN techmart_dw.payments pay ON o.order_id = pay.order_id
WHERE o.status = 'completed'
  AND o.order_date >= DATE_ADD(CURRENT_DATE(), -180)
ORDER BY o.order_date DESC
LIMIT 20

## ⚠️ JOIN Edge Cases & Gotchas

### The Duplicate Row Problem
When joining, if one side has duplicates, your result MULTIPLIES:
```
orders (1 row for order 42) JOIN payments (3 payments for order 42)
= 3 rows in result (one per payment)

If you then SUM(order_total), you get 3x the actual total!
```

**Fix:** Aggregate BEFORE joining, or use DISTINCT.

### NULL in JOIN Conditions
```sql
-- ⚠️ This will MISS rows where customer_id is NULL on either side!
SELECT * FROM orders o JOIN customers c ON o.customer_id = c.customer_id

-- NULLs never match in JOIN conditions (NULL = NULL is FALSE)
```

### Performance: JOIN Order Matters
- Put the LARGEST table first (or let the optimizer decide)
- Filter BEFORE joining when possible (reduces rows entering the join)
- Use broadcast hints for small tables: `/*+ BROADCAST(small_table) */`

---
# 📗 Section 6: Intermediate SQL

## Subqueries & EXISTS

**Three types:** in WHERE (filter), in FROM (derived table), in SELECT (scalar).

**Performance note:** Correlated subqueries execute once per row — can be slow!

In [ ]:
%sql
-- Subquery in WHERE: orders above average
SELECT order_id, customer_id, total_amount
FROM orders
WHERE total_amount > (SELECT AVG(total_amount) FROM orders)
ORDER BY total_amount DESC LIMIT 10;

In [ ]:
%sql
-- Subquery in FROM: derived table
SELECT customer_segment, ROUND(AVG(order_count),1) AS avg_orders, ROUND(AVG(total_spent),2) AS avg_spend
FROM (
  SELECT c.customer_id, c.customer_segment, COUNT(o.order_id) AS order_count, SUM(o.total_amount) AS total_spent
  FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id
  GROUP BY c.customer_id, c.customer_segment
) t
GROUP BY customer_segment ORDER BY avg_spend DESC;

In [ ]:
%sql
-- EXISTS: customers who have completed orders
SELECT c.customer_id, c.first_name, c.customer_segment
FROM customers c
WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id AND o.status = 'completed')
LIMIT 10;

## CTEs (Common Table Expressions)

**WHY:** CTEs make complex queries readable by breaking them into named steps.
Reusable, readable, and support recursion.

In [ ]:
%sql
-- Chained CTEs: RFM Analysis
WITH customer_orders AS (
  SELECT customer_id, MAX(order_date) AS last_order, COUNT(*) AS frequency, SUM(total_amount) AS monetary
  FROM orders WHERE status = 'completed' GROUP BY customer_id
),
rfm_scores AS (
  SELECT customer_id, DATEDIFF(current_date(), last_order) AS recency_days, frequency, monetary,
    NTILE(5) OVER (ORDER BY DATEDIFF(current_date(), last_order) ASC) AS r_score,
    NTILE(5) OVER (ORDER BY frequency DESC) AS f_score,
    NTILE(5) OVER (ORDER BY monetary DESC) AS m_score
  FROM customer_orders
)
SELECT customer_id, recency_days, frequency, ROUND(monetary,2) AS monetary,
  r_score, f_score, m_score, CONCAT(r_score,f_score,m_score) AS rfm_segment
FROM rfm_scores ORDER BY monetary DESC LIMIT 20;

## UNION ALL vs UNION, INTERSECT, EXCEPT

- `UNION ALL` keeps all rows (fast)
- `UNION` removes duplicates (slower)
- `INTERSECT` rows in both
- `EXCEPT` rows in first but not second

In [ ]:
%sql
SELECT 'order' AS event_type, order_id AS event_id, order_date AS event_date FROM orders WHERE order_date >= '2024-01-01'
UNION ALL
SELECT 'payment', payment_id, payment_date FROM payments WHERE payment_date >= '2024-01-01'
ORDER BY event_date DESC LIMIT 20;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: CTE Pipeline
-- ============================================
-- 1. CTE 'active_customers': customers with > 3 completed orders
-- 2. CTE 'customer_metrics': avg_order_value, days_since_first_order
-- 3. Final: top 10 by avg_order_value
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
WITH active_customers AS (
  SELECT customer_id FROM orders WHERE status='completed' GROUP BY customer_id HAVING COUNT(*)>3
),
customer_metrics AS (
  SELECT o.customer_id, ROUND(AVG(o.total_amount),2) AS avg_order_value,
    DATEDIFF(current_date(), MIN(o.order_date)) AS days_since_first
  FROM orders o JOIN active_customers ac ON o.customer_id=ac.customer_id
  WHERE o.status='completed' GROUP BY o.customer_id
)
SELECT * FROM customer_metrics ORDER BY avg_order_value DESC LIMIT 10;

---
# 📗 Section 7: Window Functions

**WHY:** Window functions compute values across related rows WITHOUT collapsing them.
Unlike GROUP BY, window functions keep all rows and add computed columns.

**Syntax:** `function() OVER (PARTITION BY ... ORDER BY ... ROWS/RANGE ...)`

**Categories:**
1. Ranking: ROW_NUMBER, RANK, DENSE_RANK, NTILE
2. Offset: LAG, LEAD, FIRST_VALUE, LAST_VALUE
3. Aggregate: SUM, AVG, COUNT with OVER

In [ ]:
%sql
-- ROW_NUMBER: deduplication pattern
WITH ranked AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY email ORDER BY customer_id ASC) AS rn
  FROM customers WHERE email IS NOT NULL
)
SELECT customer_id, first_name, last_name, email, rn
FROM ranked WHERE rn > 1 LIMIT 15;

In [ ]:
%sql
-- RANK vs DENSE_RANK
SELECT department, first_name, salary,
  RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rank_val,
  DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dense_rank_val,
  ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS row_num
FROM employees WHERE department IN ('Engineering','Sales')
ORDER BY department, salary DESC LIMIT 20;

In [ ]:
%sql
-- LAG/LEAD: month-over-month comparison
WITH monthly_revenue AS (
  SELECT date_format(order_date,'yyyy-MM') AS month, SUM(total_amount) AS revenue
  FROM orders WHERE status='completed' GROUP BY date_format(order_date,'yyyy-MM')
)
SELECT month, ROUND(revenue,2) AS revenue,
  ROUND(LAG(revenue) OVER (ORDER BY month),2) AS prev_month,
  ROUND((revenue - LAG(revenue) OVER (ORDER BY month)) / LAG(revenue) OVER (ORDER BY month) * 100, 2) AS mom_growth_pct
FROM monthly_revenue ORDER BY month LIMIT 20;

In [ ]:
%sql
-- Running total and 7-day moving average
WITH daily AS (
  SELECT order_date, COUNT(*) AS orders, SUM(total_amount) AS revenue
  FROM orders WHERE order_date BETWEEN '2023-06-01' AND '2023-06-30'
  GROUP BY order_date
)
SELECT order_date, orders, ROUND(revenue,2) AS revenue,
  ROUND(SUM(revenue) OVER (ORDER BY order_date),2) AS running_total,
  ROUND(AVG(revenue) OVER (ORDER BY order_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),2) AS moving_avg_7d
FROM daily ORDER BY order_date;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Growth Report
-- ============================================
-- For each month show:
-- 1. Revenue, 2. Previous month (LAG), 3. Growth %
-- 4. 3-month moving average
-- 5. Cumulative YTD (reset each year with PARTITION BY year)
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
WITH monthly AS (
  SELECT YEAR(order_date) AS yr, MONTH(order_date) AS mo,
    date_format(order_date,'yyyy-MM') AS month, SUM(total_amount) AS revenue
  FROM orders WHERE status='completed'
  GROUP BY YEAR(order_date), MONTH(order_date), date_format(order_date,'yyyy-MM')
)
SELECT month, ROUND(revenue,2) AS revenue,
  ROUND(LAG(revenue) OVER (ORDER BY month),2) AS prev_month,
  ROUND((revenue-LAG(revenue) OVER (ORDER BY month))/LAG(revenue) OVER (ORDER BY month)*100,2) AS growth_pct,
  ROUND(AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),2) AS ma_3m,
  ROUND(SUM(revenue) OVER (PARTITION BY yr ORDER BY mo),2) AS ytd_revenue
FROM monthly ORDER BY month;

## 🎯 Practice: Window Functions

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Window Function Challenges
-- ============================================
-- 1. Rank customers by lifetime_value within each segment
-- 2. Calculate running total of daily revenue
-- 3. Find the previous order date for each customer (LAG)
-- 4. Calculate 7-day moving average of order count
--
-- Write your queries below:


In [ ]:
%sql
-- ✅ SOLUTION 1: Rank within segment
SELECT 
    customer_id,
    CONCAT(first_name, ' ', last_name) AS name,
    customer_segment,
    lifetime_value,
    RANK() OVER (PARTITION BY customer_segment ORDER BY lifetime_value DESC) AS rank_in_segment,
    DENSE_RANK() OVER (PARTITION BY customer_segment ORDER BY lifetime_value DESC) AS dense_rank_in_segment
FROM techmart_dw.customers
WHERE is_active = true
ORDER BY customer_segment, rank_in_segment
LIMIT 20

In [ ]:
%sql
-- ✅ SOLUTION 2: Running total of daily revenue
SELECT 
    order_date,
    daily_revenue,
    SUM(daily_revenue) OVER (ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
    AVG(daily_revenue) OVER (ORDER BY order_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS moving_avg_7d
FROM (
    SELECT 
        order_date,
        ROUND(SUM(total_amount), 2) AS daily_revenue
    FROM techmart_dw.orders
    WHERE status IN ('completed', 'shipped')
    GROUP BY order_date
)
ORDER BY order_date DESC
LIMIT 30

In [ ]:
%sql
-- ✅ SOLUTION 3: Previous order date per customer (LAG)
SELECT 
    customer_id,
    order_id,
    order_date,
    LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date,
    DATEDIFF(order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date)) AS days_between_orders
FROM techmart_dw.orders
WHERE customer_id IN (1, 2, 3, 4, 5)
ORDER BY customer_id, order_date

## Window Function Frame Specification

The frame defines WHICH rows the window function considers:

```sql
-- ROWS BETWEEN: counts physical rows
SUM(amount) OVER (ORDER BY date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
-- Looks at current row + 2 rows before (exactly 3 rows)

-- RANGE BETWEEN: considers logical range of values
SUM(amount) OVER (ORDER BY date RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND CURRENT ROW)
-- Looks at all rows within 7 days before current row

-- Common frames:
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW  -- Running total (all rows up to now)
ROWS BETWEEN 6 PRECEDING AND CURRENT ROW          -- 7-day window (current + 6 before)
ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING           -- 3-row centered window
ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING   -- All rows from now to end
```

### RANK vs DENSE_RANK vs ROW_NUMBER

```
Data: [100, 90, 90, 80, 70]

ROW_NUMBER: 1, 2, 3, 4, 5  (always unique, arbitrary for ties)
RANK:       1, 2, 2, 4, 5  (ties get same rank, next rank skipped)
DENSE_RANK: 1, 2, 2, 3, 4  (ties get same rank, no gaps)
```

| Function | Use When |
|----------|----------|
| ROW_NUMBER | Deduplication (pick one row per group) |
| RANK | Competition ranking (1st, 2nd, 2nd, 4th) |
| DENSE_RANK | Dense ranking (1st, 2nd, 2nd, 3rd) |
| NTILE(4) | Divide into quartiles/buckets |

---
# 📗 Section 8: Data Engineering SQL Patterns

These are the patterns you'll use daily as a Data Engineer.

## Deduplication Pattern

ROW_NUMBER + filter is the standard deduplication approach.

In [ ]:
%sql
-- Deduplication: keep latest record per email
CREATE OR REPLACE TEMP VIEW customers_deduped AS
WITH ranked AS (
  SELECT *, ROW_NUMBER() OVER (
    PARTITION BY LOWER(COALESCE(email, CAST(customer_id AS STRING)))
    ORDER BY updated_at DESC
  ) AS rn FROM customers
)
SELECT * FROM ranked WHERE rn = 1;

SELECT COUNT(*) AS deduped_count FROM customers_deduped;

## MERGE INTO — The UPSERT Pattern

MERGE handles insert-or-update in a single atomic operation. Essential for incremental loading.

In [ ]:
%sql
-- Create staging table
DROP TABLE IF EXISTS customers_staging;
CREATE TABLE customers_staging AS SELECT * FROM customers WHERE customer_id <= 10;
UPDATE customers_staging SET lifetime_value = lifetime_value + 1000 WHERE customer_id <= 5;

INSERT INTO customers_staging
SELECT 9999,'New','Customer','new@test.com','+1-555-9999','123 New St','Seattle','WA','98101','US',
  current_date(),'New',0.00,true,'{"source":"test"}',current_timestamp(),current_timestamp();

In [ ]:
%sql
-- MERGE: upsert from staging
MERGE INTO customers AS target
USING customers_staging AS source ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET target.lifetime_value = source.lifetime_value, target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT *;

## SCD Type 2 — History Tracking

Preserves full history of changes. Each change creates a new row with effective dates.

In [ ]:
%sql
DROP TABLE IF EXISTS customers_scd2;
CREATE TABLE customers_scd2 (
  customer_sk INT, customer_id INT, first_name STRING, last_name STRING,
  customer_segment STRING, lifetime_value DECIMAL(12,2),
  effective_from DATE, effective_to DATE, is_current BOOLEAN, _row_hash STRING
);

INSERT INTO customers_scd2
SELECT ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_sk,
  customer_id, first_name, last_name, customer_segment, lifetime_value,
  CAST('2020-01-01' AS DATE) AS effective_from, CAST('9999-12-31' AS DATE) AS effective_to,
  true AS is_current,
  MD5(CONCAT(first_name,last_name,customer_segment,CAST(lifetime_value AS STRING))) AS _row_hash
FROM customers WHERE customer_id <= 100;

## Bronze → Silver → Gold Pattern

The medallion architecture: raw (Bronze), cleaned (Silver), business aggregates (Gold).

In [ ]:
%sql
-- Bronze: raw data with audit columns
DROP TABLE IF EXISTS bronze_orders;
CREATE TABLE bronze_orders AS
SELECT *, current_timestamp() AS _loaded_at, 'batch_001' AS _source_batch, 'orders_api' AS _source_system
FROM orders WHERE order_date >= '2024-01-01';
SELECT COUNT(*) AS bronze_count FROM bronze_orders;

In [ ]:
%sql
-- Silver: cleaned, deduped, validated
DROP TABLE IF EXISTS silver_orders;
CREATE TABLE silver_orders AS
WITH deduped AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY _loaded_at DESC) AS rn FROM bronze_orders
)
SELECT order_id, customer_id, order_date, status,
  CASE WHEN total_amount<0 THEN 0 ELSE total_amount END AS total_amount,
  discount_amount, shipping_cost, tax_amount, payment_method, order_source,
  _loaded_at, _source_batch, current_timestamp() AS _processed_at
FROM deduped WHERE rn=1 AND order_id IS NOT NULL AND customer_id IS NOT NULL;
SELECT COUNT(*) AS silver_count FROM silver_orders;

In [ ]:
%sql
-- Gold: business aggregates
DROP TABLE IF EXISTS gold_daily_sales;
CREATE TABLE gold_daily_sales AS
SELECT order_date, order_source, payment_method,
  COUNT(*) AS order_count, SUM(total_amount) AS total_revenue,
  AVG(total_amount) AS avg_order_value, current_timestamp() AS _aggregated_at
FROM silver_orders GROUP BY order_date, order_source, payment_method;
SELECT * FROM gold_daily_sales ORDER BY order_date DESC LIMIT 10;

## Data Quality Check Framework

In [ ]:
%sql
SELECT 'orders' AS tbl, 'order_id' AS col, COUNT(*) AS total,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS nulls,
  ROUND(SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END)/COUNT(*)*100,2) AS null_pct
FROM orders
UNION ALL
SELECT 'customers','email',COUNT(*),SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END),
  ROUND(SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END)/COUNT(*)*100,2) FROM customers;

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Incremental Pipeline
-- ============================================
-- 1. Create a watermark table
-- 2. Select NEW orders (created_at > watermark)
-- 3. Deduplicate
-- 4. MERGE into target
-- 5. Update watermark
--
-- Write your code below:


In [ ]:
%sql
-- ============================================
-- ✅ SOLUTION
-- ============================================
DROP TABLE IF EXISTS pipeline_watermark;
CREATE TABLE pipeline_watermark (pipeline_name STRING, last_processed_at TIMESTAMP);
INSERT INTO pipeline_watermark VALUES ('orders_pipeline', '2020-01-01');

DROP TABLE IF EXISTS orders_target;
CREATE TABLE orders_target AS SELECT * FROM orders WHERE 1=0;

MERGE INTO orders_target AS target
USING (
  WITH new_recs AS (
    SELECT * FROM orders WHERE created_at > (SELECT last_processed_at FROM pipeline_watermark WHERE pipeline_name='orders_pipeline')
  ),
  deduped AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY created_at DESC) AS rn FROM new_recs)
  SELECT * FROM deduped WHERE rn=1
) AS source ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

UPDATE pipeline_watermark SET last_processed_at=current_timestamp() WHERE pipeline_name='orders_pipeline';

## 🎯 Practice: Data Engineering SQL

In [ ]:
%sql
-- ============================================
-- 🎯 YOUR TURN: Build an Incremental Pipeline
-- ============================================
-- Scenario: You need to incrementally load NEW orders into a silver table.
-- 
-- Steps:
-- 1. Create a silver_orders table (if not exists)
-- 2. Find the max order_date already in silver (watermark)
-- 3. MERGE new orders from bronze (orders table) into silver
-- 4. Add audit columns: _loaded_at, _source
--
-- Write your MERGE statement below:


In [ ]:
%sql
-- ✅ SOLUTION: Incremental MERGE with audit columns
-- Step 1: Create silver table
CREATE TABLE IF NOT EXISTS techmart_dw.silver_orders_demo (
    order_id INT,
    customer_id INT,
    order_date DATE,
    status STRING,
    total_amount DECIMAL(12,2),
    payment_method STRING,
    _loaded_at TIMESTAMP,
    _source STRING,
    _row_hash STRING
) USING DELTA

In [ ]:
%sql
-- Step 2 & 3: MERGE with watermark pattern
MERGE INTO techmart_dw.silver_orders_demo AS target
USING (
    SELECT 
        order_id,
        customer_id,
        order_date,
        LOWER(TRIM(status)) AS status,
        total_amount,
        payment_method,
        current_timestamp() AS _loaded_at,
        'techmart_dw.orders' AS _source,
        md5(concat_ws('|', order_id, customer_id, order_date, status, total_amount)) AS _row_hash
    FROM techmart_dw.orders
    WHERE order_date >= COALESCE(
        (SELECT MAX(order_date) FROM techmart_dw.silver_orders_demo),
        '1900-01-01'
    )
) AS source
ON target.order_id = source.order_id
WHEN MATCHED AND target._row_hash != source._row_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sql
-- Verify the incremental load
SELECT 
    COUNT(*) AS total_rows,
    MIN(order_date) AS earliest_order,
    MAX(order_date) AS latest_order,
    MAX(_loaded_at) AS last_loaded
FROM techmart_dw.silver_orders_demo

## Data Quality Check Framework

Every production pipeline needs quality gates between layers:

```sql
-- Quality check pattern: run AFTER each layer, BEFORE promoting to next
-- If any check fails → alert, don't promote to Gold

-- Check 1: Completeness (no unexpected NULLs)
SELECT 'null_check' AS check_name,
       COUNT(*) AS failures
FROM silver_table
WHERE required_column IS NULL;

-- Check 2: Uniqueness (no duplicates on primary key)
SELECT 'duplicate_check' AS check_name,
       COUNT(*) - COUNT(DISTINCT primary_key) AS failures
FROM silver_table;

-- Check 3: Referential integrity (FKs exist in parent)
SELECT 'fk_check' AS check_name,
       COUNT(*) AS failures
FROM silver_orders o
LEFT JOIN silver_customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

-- Check 4: Range validity
SELECT 'range_check' AS check_name,
       COUNT(*) AS failures
FROM silver_table
WHERE amount < 0 OR amount > 1000000;
```

In [ ]:
%sql
-- Run quality checks on our silver table
SELECT 'null_customer_id' AS check_name, COUNT(*) AS failures
FROM techmart_dw.silver_orders_demo WHERE customer_id IS NULL
UNION ALL
SELECT 'duplicate_order_id', COUNT(*) - COUNT(DISTINCT order_id)
FROM techmart_dw.silver_orders_demo
UNION ALL
SELECT 'negative_amount', COUNT(*)
FROM techmart_dw.silver_orders_demo WHERE total_amount < 0
UNION ALL
SELECT 'future_dates', COUNT(*)
FROM techmart_dw.silver_orders_demo WHERE order_date > CURRENT_DATE()

---
# 📗 Section 9: Performance & Optimization

## EXPLAIN Plans

In [ ]:
%sql
EXPLAIN
SELECT c.customer_segment, COUNT(*) AS cnt, SUM(o.total_amount) AS revenue
FROM customers c JOIN orders o ON c.customer_id = o.customer_id
WHERE o.status = 'completed' GROUP BY c.customer_segment;

In [ ]:
%sql
-- OPTIMIZE: compact small files
OPTIMIZE orders;

In [ ]:
%sql
DESCRIBE HISTORY orders LIMIT 5;

In [ ]:
%sql
-- CACHE for repeated queries
CACHE TABLE orders;
SELECT status, COUNT(*) FROM orders GROUP BY status;
UNCACHE TABLE orders;

## Performance Best Practices

| Practice | Why |
|----------|-----|
| Select only needed columns | Less I/O |
| Filter early | Process fewer rows |
| Use partition columns in WHERE | Skip file groups |
| Avoid SELECT DISTINCT on large tables | Full shuffle |
| Use EXISTS over IN for large subqueries | Short-circuits |
| Liquid Clustering for new tables | Better than partitioning + ZORDER |

---
# 🏆 Section 10: Interview Challenges

## Challenge 1: Gaps and Islands (Consecutive Days)
Find customers who ordered on 3+ consecutive days.

In [ ]:
%sql
WITH daily_orders AS (
  SELECT DISTINCT customer_id, order_date FROM orders WHERE status='completed'
),
islands AS (
  SELECT customer_id, order_date,
    date_sub(order_date, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date)) AS island_id
  FROM daily_orders
),
streaks AS (
  SELECT customer_id, island_id, COUNT(*) AS consecutive_days,
    MIN(order_date) AS streak_start, MAX(order_date) AS streak_end
  FROM islands GROUP BY customer_id, island_id HAVING COUNT(*)>=3
)
SELECT * FROM streaks ORDER BY consecutive_days DESC LIMIT 15;

## Challenge 2: Customer Cohort Retention
For each monthly cohort, what % return in subsequent months?

In [ ]:
%sql
WITH first_order AS (
  SELECT customer_id, MIN(date_format(order_date,'yyyy-MM')) AS cohort_month
  FROM orders WHERE status='completed' GROUP BY customer_id
),
monthly_activity AS (
  SELECT DISTINCT customer_id, date_format(order_date,'yyyy-MM') AS activity_month
  FROM orders WHERE status='completed'
),
cohort_data AS (
  SELECT f.cohort_month, m.activity_month,
    CAST(MONTHS_BETWEEN(CAST(CONCAT(m.activity_month,'-01') AS DATE), CAST(CONCAT(f.cohort_month,'-01') AS DATE)) AS INT) AS months_since,
    COUNT(DISTINCT f.customer_id) AS customers
  FROM first_order f JOIN monthly_activity m ON f.customer_id=m.customer_id
  GROUP BY f.cohort_month, m.activity_month
)
SELECT cohort_month, months_since, customers,
  FIRST_VALUE(customers) OVER (PARTITION BY cohort_month ORDER BY months_since) AS cohort_size,
  ROUND(customers*100.0/FIRST_VALUE(customers) OVER (PARTITION BY cohort_month ORDER BY months_since),1) AS retention_pct
FROM cohort_data WHERE months_since BETWEEN 0 AND 6
ORDER BY cohort_month, months_since LIMIT 30;

## Challenge 3: Sessionization
Group website events into sessions (30-min inactivity = new session).

In [ ]:
%sql
WITH events_gap AS (
  SELECT event_id, customer_id, event_type, event_timestamp,
    CASE WHEN LAG(event_timestamp) OVER (PARTITION BY customer_id ORDER BY event_timestamp) IS NULL THEN 1
         WHEN TIMESTAMPDIFF(MINUTE, LAG(event_timestamp) OVER (PARTITION BY customer_id ORDER BY event_timestamp), event_timestamp) > 30 THEN 1
         ELSE 0 END AS new_session
  FROM website_events WHERE customer_id IS NOT NULL AND customer_id <= 10
),
sessions AS (
  SELECT *, SUM(new_session) OVER (PARTITION BY customer_id ORDER BY event_timestamp) AS session_num
  FROM events_gap
)
SELECT customer_id, session_num, COUNT(*) AS events, MIN(event_timestamp) AS start_time, MAX(event_timestamp) AS end_time
FROM sessions GROUP BY customer_id, session_num
ORDER BY customer_id, session_num LIMIT 20;

## Challenge 4: YoY Growth by Category

In [ ]:
%sql
WITH category_yearly AS (
  SELECT p.category, YEAR(o.order_date) AS yr, SUM(oi.line_total) AS revenue
  FROM order_items oi
  JOIN products p ON oi.product_id=p.product_id
  JOIN orders o ON oi.order_id=o.order_id
  WHERE o.status='completed' GROUP BY p.category, YEAR(o.order_date)
)
SELECT category, yr, ROUND(revenue,2) AS revenue,
  ROUND(LAG(revenue) OVER (PARTITION BY category ORDER BY yr),2) AS prev_year,
  ROUND((revenue-LAG(revenue) OVER (PARTITION BY category ORDER BY yr))/LAG(revenue) OVER (PARTITION BY category ORDER BY yr)*100,2) AS yoy_pct
FROM category_yearly ORDER BY category, yr;

## Challenge 5: Market Basket Analysis
Find products frequently bought together.

In [ ]:
%sql
WITH order_products AS (SELECT DISTINCT order_id, product_id FROM order_items)
SELECT a.product_id AS product_a, b.product_id AS product_b, COUNT(*) AS bought_together
FROM order_products a
JOIN order_products b ON a.order_id=b.order_id AND a.product_id < b.product_id
GROUP BY a.product_id, b.product_id
HAVING COUNT(*)>=5
ORDER BY bought_together DESC LIMIT 20;

## Challenge 6: Second Highest Salary per Department

In [ ]:
%sql
WITH ranked AS (
  SELECT department, first_name, salary,
    DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS rnk
  FROM employees WHERE is_active=true
)
SELECT department, first_name, salary FROM ranked WHERE rnk=2 ORDER BY department;

## 🎯 Bonus Challenge: RFM Analysis

**RFM (Recency, Frequency, Monetary)** is a customer segmentation technique:
- **Recency**: How recently did they purchase?
- **Frequency**: How often do they purchase?
- **Monetary**: How much do they spend?

In [ ]:
%sql
-- RFM Analysis — segment customers by behavior
WITH rfm_raw AS (
    SELECT 
        c.customer_id,
        CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
        DATEDIFF(CURRENT_DATE(), MAX(o.order_date)) AS recency_days,
        COUNT(DISTINCT o.order_id) AS frequency,
        ROUND(SUM(o.total_amount), 2) AS monetary
    FROM techmart_dw.customers c
    JOIN techmart_dw.orders o ON c.customer_id = o.customer_id
    WHERE o.status IN ('completed', 'shipped')
    GROUP BY c.customer_id, c.first_name, c.last_name
),
rfm_scored AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY recency_days ASC) AS r_score,   -- Lower recency = better
        NTILE(5) OVER (ORDER BY frequency DESC) AS f_score,      -- Higher frequency = better
        NTILE(5) OVER (ORDER BY monetary DESC) AS m_score        -- Higher monetary = better
    FROM rfm_raw
)
SELECT 
    customer_id,
    customer_name,
    recency_days,
    frequency,
    monetary,
    r_score, f_score, m_score,
    CONCAT(r_score, f_score, m_score) AS rfm_segment,
    CASE 
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
        WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal Customers'
        WHEN r_score >= 4 AND f_score <= 2 THEN 'New Customers'
        WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
        WHEN r_score <= 2 AND f_score <= 2 THEN 'Lost'
        ELSE 'Needs Attention'
    END AS customer_segment
FROM rfm_scored
ORDER BY monetary DESC
LIMIT 20

---
# 🚀 Section 11: Mini Projects

## Project 1: Customer 360 View

In [ ]:
%sql
DROP TABLE IF EXISTS customer_360;
CREATE TABLE customer_360 AS
WITH order_metrics AS (
  SELECT customer_id, COUNT(*) AS total_orders,
    SUM(CASE WHEN status='completed' THEN 1 ELSE 0 END) AS completed,
    SUM(total_amount) AS total_revenue, AVG(total_amount) AS avg_order_value,
    MIN(order_date) AS first_order, MAX(order_date) AS last_order,
    DATEDIFF(current_date(), MAX(order_date)) AS days_since_last
  FROM orders GROUP BY customer_id
),
payment_prefs AS (
  SELECT customer_id, payment_method, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY COUNT(*) DESC) AS rn
  FROM orders GROUP BY customer_id, payment_method
),
web AS (
  SELECT customer_id, COUNT(*) AS events, COUNT(DISTINCT session_id) AS sessions
  FROM website_events WHERE customer_id IS NOT NULL GROUP BY customer_id
)
SELECT c.customer_id, c.first_name, c.last_name, c.customer_segment, c.is_active,
  COALESCE(om.total_orders,0) AS total_orders, COALESCE(om.completed,0) AS completed_orders,
  COALESCE(ROUND(om.total_revenue,2),0) AS total_revenue,
  COALESCE(ROUND(om.avg_order_value,2),0) AS avg_order_value,
  om.first_order, om.last_order, om.days_since_last,
  pp.payment_method AS preferred_payment,
  COALESCE(w.sessions,0) AS web_sessions
FROM customers c
LEFT JOIN order_metrics om ON c.customer_id=om.customer_id
LEFT JOIN payment_prefs pp ON c.customer_id=pp.customer_id AND pp.rn=1
LEFT JOIN web w ON c.customer_id=w.customer_id;

SELECT * FROM customer_360 ORDER BY total_revenue DESC LIMIT 10;

## Project 2: Sales Dashboard Gold Table

In [ ]:
%sql
DROP TABLE IF EXISTS gold_sales_dashboard;
CREATE TABLE gold_sales_dashboard AS
SELECT o.order_date, YEAR(o.order_date) AS year, MONTH(o.order_date) AS month,
  p.category, c.customer_segment, c.state, o.order_source, o.payment_method,
  COUNT(DISTINCT o.order_id) AS orders, COUNT(DISTINCT o.customer_id) AS unique_customers,
  SUM(oi.quantity) AS units_sold, ROUND(SUM(oi.line_total),2) AS revenue
FROM orders o
JOIN customers c ON o.customer_id=c.customer_id
JOIN order_items oi ON o.order_id=oi.order_id
JOIN products p ON oi.product_id=p.product_id
WHERE o.status='completed'
GROUP BY o.order_date, YEAR(o.order_date), MONTH(o.order_date),
  p.category, c.customer_segment, c.state, o.order_source, o.payment_method;

SELECT category, SUM(revenue) AS total_rev FROM gold_sales_dashboard GROUP BY category ORDER BY total_rev DESC;

## Project 3: Error Monitoring Pipeline

In [ ]:
%sql
DROP TABLE IF EXISTS gold_error_monitoring;
CREATE TABLE gold_error_monitoring AS
SELECT log_date, service_name, log_level, message, COUNT(*) AS occurrences
FROM app_logs WHERE log_level IN ('ERROR','WARN')
GROUP BY log_date, service_name, log_level, message;

SELECT service_name, SUM(occurrences) AS total_errors
FROM gold_error_monitoring WHERE log_level='ERROR'
GROUP BY service_name ORDER BY total_errors DESC;

---
# 🎓 Section 12: Capstone Project

## Full Bronze → Silver → Gold Pipeline with Quality Gates

**Scenario:** New payment data arrives daily. Build a complete pipeline.

In [ ]:
%sql
-- Bronze: raw ingestion
DROP TABLE IF EXISTS capstone_bronze;
CREATE TABLE capstone_bronze AS
SELECT *, current_timestamp() AS _ingested_at, 'payments_20240601.csv' AS _source_file, 'batch_001' AS _batch_id
FROM payments WHERE payment_date >= '2024-01-01';

In [ ]:
%sql
-- Silver: clean, validate, dedupe
DROP TABLE IF EXISTS capstone_silver;
CREATE TABLE capstone_silver AS
WITH checks AS (
  SELECT *,
    CASE WHEN payment_id IS NULL THEN 'FAIL' ELSE 'PASS' END AS chk_pk,
    CASE WHEN amount<=0 THEN 'FAIL' ELSE 'PASS' END AS chk_amount,
    ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY _ingested_at DESC) AS rn
  FROM capstone_bronze
)
SELECT payment_id, order_id, payment_date, amount,
  LOWER(TRIM(payment_method)) AS payment_method,
  LOWER(TRIM(payment_status)) AS payment_status,
  transaction_ref, gateway_response, _ingested_at, _source_file,
  current_timestamp() AS _processed_at,
  MD5(CONCAT(CAST(payment_id AS STRING),CAST(amount AS STRING),payment_status)) AS _row_hash
FROM checks WHERE rn=1 AND chk_pk='PASS' AND chk_amount='PASS';

SELECT COUNT(*) AS silver_rows FROM capstone_silver;

In [ ]:
%sql
-- Gold: business aggregates
DROP TABLE IF EXISTS capstone_gold;
CREATE TABLE capstone_gold AS
SELECT payment_date, payment_method, payment_status,
  COUNT(*) AS txn_count, SUM(amount) AS total_amount,
  AVG(amount) AS avg_amount, MIN(amount) AS min_amount, MAX(amount) AS max_amount,
  current_timestamp() AS _aggregated_at
FROM capstone_silver GROUP BY payment_date, payment_method, payment_status;

SELECT payment_method, SUM(txn_count) AS txns, ROUND(SUM(total_amount),2) AS revenue
FROM capstone_gold WHERE payment_status='completed'
GROUP BY payment_method ORDER BY revenue DESC;

In [ ]:
%sql
-- Pipeline quality report
SELECT 'bronze' AS layer, COUNT(*) AS rows FROM capstone_bronze
UNION ALL SELECT 'silver', COUNT(*) FROM capstone_silver
UNION ALL SELECT 'gold', COUNT(*) FROM capstone_gold;

---
# 📋 Section 13: Summary & Verification

In [ ]:
%sql
-- Final verification: all tables
SELECT 'customers' AS tbl, COUNT(*) AS rows FROM customers
UNION ALL SELECT 'products', COUNT(*) FROM products
UNION ALL SELECT 'orders', COUNT(*) FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL SELECT 'payments', COUNT(*) FROM payments
UNION ALL SELECT 'employees', COUNT(*) FROM employees
UNION ALL SELECT 'inventory', COUNT(*) FROM inventory
UNION ALL SELECT 'shipments', COUNT(*) FROM shipments
UNION ALL SELECT 'website_events', COUNT(*) FROM website_events
UNION ALL SELECT 'app_logs', COUNT(*) FROM app_logs
UNION ALL SELECT 'transactions', COUNT(*) FROM transactions
UNION ALL SELECT 'customer_360', COUNT(*) FROM customer_360
UNION ALL SELECT 'gold_sales_dashboard', COUNT(*) FROM gold_sales_dashboard
UNION ALL SELECT 'gold_error_monitoring', COUNT(*) FROM gold_error_monitoring
UNION ALL SELECT 'capstone_bronze', COUNT(*) FROM capstone_bronze
UNION ALL SELECT 'capstone_silver', COUNT(*) FROM capstone_silver
UNION ALL SELECT 'capstone_gold', COUNT(*) FROM capstone_gold
ORDER BY tbl;

In [ ]:
%sql
SHOW TABLES IN techmart_dw;

## ✅ Notebook Complete!

**What was covered:**
- 11 core tables with 300,000+ total rows
- Intentional data quality issues for practice
- SQL fundamentals: SELECT, WHERE, JOIN, GROUP BY
- Intermediate: CTEs, subqueries, set operations
- Advanced: Window functions, MERGE, SCD Type 2
- Data Engineering: Bronze/Silver/Gold, incremental loads
- Performance: EXPLAIN, OPTIMIZE, caching
- 6 interview challenges
- 3 mini projects
- 1 full capstone pipeline

**Next:** Notebook 02 — Python Foundations for Data Engineering

---
*All tables remain in `techmart_dw` for use in subsequent notebooks.*